In [1]:
import numpy as np
import scipy.stats as stats
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
PUBLIC_SRC = 'https://seshat.datasd.org/get_it_done_reports'
YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
PRIVATE_CSV = '../data/gid_cases-001.csv'
ATTACH_CSV = '../data/public-attachments-cases.csv'
REFFERALS_CSV = '../data/referral data for UCSD-2026-11-02-23-09-16.csv'
WINDOW_START, WINDOW_END = '2016-01-01', '2026-11-01'

In [3]:
closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv') for y in YEARS], ignore_index=True)
open_ = pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_open_datasd.csv')

C:\Users\cjjen\AppData\Local\Temp\ipykernel_16092\1658780389.py:1: DtypeWarning: Columns (7,20) have mixed types. Specify dtype option on import or set low_memory=False.
  closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv') for y in YEARS], ignore_index=True)
C:\Users\cjjen\AppData\Local\Temp\ipykernel_16092\1658780389.py:1: DtypeWarning: Columns (2,17) have mixed types. Specify dtype option on import or set low_memory=False.
  closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv') for y in YEARS], ignore_index=True)
C:\Users\cjjen\AppData\Local\Temp\ipykernel_16092\1658780389.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv') for y in YEARS], ignore_index=True)
C:\Users\cjjen\AppData\Local\Temp\ipykernel_16092\1658780389.py:1: DtypeWarning: Columns (2) have mixed

In [4]:
pub = pd.concat([closed, open_], ignore_index=True).drop_duplicates('service_request_id')

pub['date_requested'] = pd.to_datetime(pub['date_requested'])
pub['date_requested'] = pub['date_requested'].dt.tz_localize('UTC').dt.tz_convert('US/Pacific')
pub['date_closed'] = pd.to_datetime(pub['date_closed'])

pub = pub[(pub['date_requested'] >= WINDOW_START) & (pub['date_requested'] < WINDOW_END)].copy()

In [ ]:
PRIV_COLS = [
    'Case Reason', 'Last Modified Date', 'Parent Case ID', 'Parent Case ID', 'Case Number',
    'Owner ID', 'Last Modified By ID', 'Created By ID', 'Created Date',
    ]

RENAME = {t:t.lower().replace(' ', '_') for t in PRIV_COLS}
RENAME.update({'Status': 'private_status'})
priv = pd.read_csv(PRIVATE_CSV, usecols=list(RENAME), dtype=str).rename(columns=RENAME)

In [ ]:
priv['case_number'] = pd.to_numeric(priv['case_number'], errors='coerce')
priv = priv.dropna(subset=['case_number'])
priv['case_number'] = priv['case_number'].astype(int)
priv.drop_duplicates('case_number', inplace=True)

priv['last_modified_date'] = pd.to_datetime(priv['last_modified_date'])
priv['owner_type'] = priv['owner_id'].str[:3].map({'005': 'worker', '00G': 'queue'}).fillna('other')
priv['modified_by_worker'] = priv['last_modified_by_id'].str[:3].eq('005')
priv['case_number'] = priv['case_number'].astype(float)
priv.drop(columns=['owner_id', 'last_modified_by_id'], inplace=True)
priv.dropna(axis=1, how='all', inplace=True)

In [ ]:
datasd = pub.merge(priv, left_on='service_request_id', right_on='case_number', how='left')

In [ ]:
public_drop = [
    'service_request_parent_id',
    'sap_notification_number',  # Keep if using it 
    'comm_plan_code',
    'comm_plan_name',
    'iamfloc',
    'floc'
]

pub = pub.drop(columns=public_drop, errors='ignore')

In [ ]:
merged = pub.merge(
    priv,
    left_on='service_request_id',
    right_on='case_number',
    how='left',
    suffixes=('_public', '_private')
)

In [28]:
closed = merged[merged['closed'] == 'TRUE']

cause_code = Stormwater
parking_violation_type = parking
problem_category_image = stormwater
problem_type = vegetation
specify_the_issue = illegal dumping
violation_name , violation_type = building issue